In [ ]:
from src.mineru_parser import MinerUParser
from src.cybersec_extractor import CybersecEntityExtractor
from src.neo4j_manager import Neo4jManager
from src.hybrid_retrieval import HybridRetriever

c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Parse document
parser_config = {"dtype": "auto", "device_map": "auto"}
parser = MinerUParser(parser_config)
parsed_doc = parser.parse("data/WickedRose_andNCPH.pdf")

Predict: 100%|██████████| 7/7 [00:32<00:00,  4.66s/it]
2025-10-29 18:09:51,466 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page2_0_NCHP 50 Screenshot GinWui Rootkit.png
2025-10-29 18:09:51,483 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page3_1_image_1.png
2025-10-29 18:09:51,510 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page5_2_image_2.png
2025-10-29 18:09:51,527 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page6_3_Wicked Roses Website wwwmghackercom.png
2025-10-29 18:09:51,542 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page6_4_NCPH Studio website wwwncphnet.png
2025-10-29 18:09:51,595 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_andNCPH_page7_5_Zigong Sichuan Province in south-central China.png
2025-10-29 18:09:51,632 - MinerUParser - INFO - Saved image: data\extracted_images\WickedRose_and

In [3]:
extractor_config = {
    "spacy_model": "en_core_web_lg"
}
extractor = CybersecEntityExtractor(extractor_config)

# Extract from your parsed document
entities = extractor.extract_entities(parsed_doc.text)
relations = extractor.extract_relations(parsed_doc.text, entities)

2025-10-29 18:09:52,717 - CybersecEntityExtractor - INFO - Loaded SpaCy model: en_core_web_lg
2025-10-29 18:09:53,309 - CybersecEntityExtractor - INFO - Extracted 598 entities
2025-10-29 18:09:53,332 - CybersecEntityExtractor - INFO - Extracted 447 relations


In [6]:
neo4j_manager = Neo4jManager(
    uri="neo4j://127.0.0.1:7687",
    username="neo4j",
    password="test1234"
)

2025-10-29 18:10:18,401 - Neo4jManager - INFO - Connected to Neo4j at neo4j://127.0.0.1:7687
2025-10-29 18:10:18,576 - Neo4jManager - INFO - Neo4j schema initialized


In [7]:
entity_stats = neo4j_manager.ingest_entities(entities)
relation_stats = neo4j_manager.ingest_relations(relations)
stats = neo4j_manager.get_statistics()
print(f"Knowledge Graph Stats: {stats}")

2025-10-29 18:10:20,619 - Neo4jManager - INFO - Ingested 598 entities
2025-10-29 18:10:21,607 - Neo4jManager - INFO - Ingested 447 relations


Knowledge Graph Stats: {'total_entities': 218, 'total_relations': 184, 'entity_types': [{'type': 'DATE', 'count': 60}, {'type': 'ORGANIZATION', 'count': 31}, {'type': 'MALWARE', 'count': 24}, {'type': 'NUMBER', 'count': 23}, {'type': 'DOMAIN', 'count': 17}, {'type': 'PERSON', 'count': 11}, {'type': 'LOCATION', 'count': 10}, {'type': 'SYSTEM', 'count': 10}, {'type': 'MONEY', 'count': 7}, {'type': 'URL', 'count': 5}, {'type': 'ORDINAL', 'count': 4}, {'type': 'NORP', 'count': 3}, {'type': 'ATTACK_PATTERN', 'count': 3}, {'type': 'WORK_OF_ART', 'count': 2}, {'type': 'TIME', 'count': 2}, {'type': 'EMAIL', 'count': 2}, {'type': 'EVENT', 'count': 1}, {'type': 'FAC', 'count': 1}, {'type': 'PORT', 'count': 1}, {'type': 'MS_VULN', 'count': 1}], 'relation_types': [{'type': 'EXPLOITS', 'count': 63}, {'type': 'CREATED_BY', 'count': 63}, {'type': 'USES', 'count': 37}, {'type': 'OCCURRED_ON', 'count': 16}, {'type': 'MEMBER_OF', 'count': 5}]}


In [8]:
malware_query = """
MATCH (m:MALWARE)
RETURN m.text as name, m.confidence as confidence
ORDER BY m.confidence DESC
LIMIT 10
"""
malware = neo4j_manager.query_graph(malware_query)
print(f"\nTop Malware: {malware}")

# Close connection
neo4j_manager.close()

2025-10-29 18:10:21,840 - Neo4jManager - INFO - Neo4j connection closed



Top Malware: [{'name': 'several', 'confidence': 0.92}, {'name': 'by', 'confidence': 0.92}, {'name': 'rootkit', 'confidence': 0.88}, {'name': 'exploit', 'confidence': 0.88}, {'name': 'Backdoor Rootkit', 'confidence': 0.88}, {'name': 'GinWui Backdoor Rootkit', 'confidence': 0.88}, {'name': 'Trojan', 'confidence': 0.88}, {'name': 'GinWui backdoor Trojan', 'confidence': 0.88}, {'name': 'backdoor Trojan', 'confidence': 0.88}, {'name': 'GinWui', 'confidence': 0.88}]


In [9]:
# Ollama configuration
retriever_config = {
    'model_name': 'llama3.1:8b',
    'embedding_model': 'nomic-embed-text',
    'ollama_base_url': 'http://localhost:11434'
}

retriever = HybridRetriever(neo4j_manager, retriever_config)

2025-10-29 18:10:24,387 - HybridRetriever - WARNING - Could not initialize Cypher chain: 1 validation error for GraphCypherQAChain
graph
  Input should be an instance of GraphStore [type=is_instance_of, input_value=<langchain_neo4j.graphs.n...t at 0x000001D7D5C076E0>, input_type=Neo4jGraph]
    For further information visit https://errors.pydantic.dev/2.11/v/is_instance_of
2025-10-29 18:10:27,944 - HybridRetriever - INFO - Vector store initialized with Ollama embeddings
2025-10-29 18:10:27,944 - HybridRetriever - INFO - LangChain components initialized with Ollama model: llama3.1:8b


In [10]:
# Ask questions
questions = [
    "What malware is mentioned in the documents?",
    "What is GinWui?",
    "Who created GinWui?",
    "What organizations are involved in these attacks?",
    "Describe the attack timeline"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    answer = retriever.answer_question(question, method="graph")
    print(f"A: {answer}\n")


Q: What malware is mentioned in the documents?
A: Based on the provided knowledge graph data, the following malware types are mentioned:

1. Backdoor Rootkit (confidence level: 0.88)
2. Booli (confidence level: 0.88)
3. Daserf (confidence level: 0.88)

Additionally, there is a mention of "malware" in general terms with a high confidence level (0.92), indicating the presence of malware but not specifying a particular type.

It's worth noting that the term "by" and "several" are also mentioned in relation to malware, but they do not specify specific types of malware.


Q: What is GinWui?
A: GinWui backdoor Trojan is a MALWARE.


Q: Who created GinWui?
A: Based on the knowledge graph data, there is no information available about who created GinWui. The entity "GinWui" has an entry as both a malware and an organization, but its creators are not specified in any of the provided context.


Q: What organizations are involved in these attacks?
A: Based on the provided knowledge graph data, th